# NCH-042 · A compact colour-reversing pair in Euclid Deep Field North
**Biswajit Jana**

Select the highest-ranked VIS/J reversal candidates from a held-out set of 1,135 image pairs. Use calibrated mosaics, three separate Euclid visits, Subaru HSC imaging, and public catalogue checks to evaluate the lead. The original source detections are already in MER.

The two imaging stages read about 432 MB of FITS ranges. Catalogue and spectrum steps download additional data. The results and image are available on GitHub without rerunning.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

branch = "research/euclid-dr1-foundation-day01-20261114"
checkout = Path("daily-astro-notebooks").resolve()
relative = Path("euclid-dr1-foundation/edf-north-compact-doubles")
if not checkout.exists():
    subprocess.run(
        ["git", "clone", "--branch", branch, "--depth", "1", "--filter=blob:none", "--sparse",
         "https://github.com/Biswajit1999/daily-astro-notebooks.git", str(checkout)],
        check=True,
    )
subprocess.run(["git", "-C", str(checkout), "sparse-checkout", "set", str(relative)], check=True)
study = checkout / relative
cache = Path("holdout-colour-cache").resolve()
env = {**os.environ, "OPENBLAS_NUM_THREADS": "1", "MPLBACKEND": "Agg"}
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(study / "requirements.txt")], check=True)


## Select and register calibrated cutouts


In [ ]:
def run(script, *params):
    subprocess.run([sys.executable, str(study / script), *map(str, params)], env=env, check=True)

run("holdout_colours.py", "--cache", cache)
run("measure_holdout_colours.py", "--cache", cache)


## Repeat the acquisition in separate visits


In [ ]:
import json

run("holdout_visits.py", "--cache", cache)
visits = json.loads((cache / "visit_checks.json").read_text())
for obs_id, bands in visits["visits"].items():
    print(obs_id, {b: round(r["apertures"]["0.28"]["b_minus_a_aperture_mag"], 3) for b, r in bands.items()})


## Check released source entries and slitless spectroscopy


In [ ]:
run("holdout_catalogues.py", "--cache", cache)
spectrum = cache / "spectrum"
run("spectra.py", "--results", spectrum, "--cache", spectrum)
info = json.loads((spectrum / "spectral_checks.json").read_text())
for check in info["checks"]:
    measurable = {k: round(v["snr"], 2) for k, v in check.get("fixed_windows", {}).items() if "snr" in v}
    print(check["object_id"], check["grism"], check.get("clean_bins", 0), measurable)


## Compare the original images and repeat measurements


In [ ]:
from IPython.display import Image, display

figure = cache / "NCH-042.png"
run("plot_holdout_lead.py", "--cache", cache, "--output", figure)
display(Image(filename=str(figure)))
